# 03 — Gold: dim_geography

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_geography` |
| **Grain** | One row per GlobalFinancialGeographyId |
| **Source** | `ref.FinancialGeographyHierarchy` |
| **PK** | `GlobalFinancialGeographyId` (int) |
| **Rows** | 1,082 |

**Hierarchy**: Region (4) → SubRegion (15) → CountryGroup (16) → Cluster (39) → Country (127) → Division (9) → Market (73) → Office (512) → Location (1,081)

> LOB columns are completely empty in the source — dropped.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_geography"
SOURCE_TABLE = "ref.FinancialGeographyHierarchy"

print(f"Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Drop LOBId, LOBCode, LOBName (all CONSTANT — 0 distinct values)
- Drop ETL dates, all Id columns (keep names + codes for BI)
- Keep GlobalFinancialGeographyId as PK

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("GlobalFinancialGeographyId").cast("int"),
    F.col("RegionCode").cast("string"),
    F.col("Region").cast("string"),
    F.col("SubRegionCode").cast("string"),
    F.col("SubRegion").cast("string"),
    F.col("CountryGroupCode").cast("string"),
    F.col("CountryGroup").cast("string"),
    F.col("ClusterCode").cast("string"),
    F.col("Cluster").cast("string"),
    F.col("CountryCode").cast("string"),
    F.col("Country").cast("string"),
    F.col("DivisionCode").cast("string"),
    F.col("Division").cast("string"),
    F.col("MarketCode").cast("string"),
    F.col("Market").cast("string"),
    F.col("OfficeCode").cast("string"),
    F.col("Office").cast("string"),
    F.col("LocationCode").cast("string"),
    F.col("Location").cast("string"),
    F.col("IsDeleted").cast("boolean")
)

# Filter out deleted Unknown members (ID = -1 AND IsDeleted = True)
before_count = df_clean.count()
df_clean = df_clean.filter(~((F.col("GlobalFinancialGeographyId") == -1) & (F.col("IsDeleted") == True)))
after_count = df_clean.count()

if before_count > after_count:
    print(f"Filtered out {before_count - after_count} deleted Unknown member(s) (ID=-1, IsDeleted=True)")

print(f"After column select + filter: {after_count:,} rows × {len(df_clean.columns)} cols")

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================

# Check if -1 still exists (meaning it had actual data and wasn't filtered)
existing_ids = df_clean.select("GlobalFinancialGeographyId").distinct().collect()
existing_id_set = {row.GlobalFinancialGeographyId for row in existing_ids}

unknown_id = -1 if -1 not in existing_id_set else -999

if unknown_id == -999:
    print(f"Using -999 for Unknown member (ID=-1 has actual data)")

unknown_row = spark.createDataFrame([(
    unknown_id,
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", False
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"Added Unknown member (ID={unknown_id}): {df_final.count():,} rows")

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("GlobalFinancialGeographyId").distinct().count()

print(f"DQ Checks")
print(f"   Total rows:     {total:,}")
print(f"   Duplicate PKs:  {dupes}")
assert dupes == 0, f"ERROR: Duplicates found!"
print("\nAll DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")